# trapX LLM Advisory — **Any date + time** (JSONL + log, no broker APIs)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/advisory_any_bar_colab.ipynb)

Pick a **DATE** and **BAR_TIME**; this reads the verdict for that bar from your Google Drive day-folder **`VM-4-logs/<Mon_DD>/`** (e.g. `2026-06-05 -> Jun_05`).

**No Kite/Breeze keys. No Postgres.** It resolves the verdict in two tiers:
1. **JSONL** `llm_advisory_<DATE>.jsonl` -> full snapshot + verdict. `MODE="replay"` shows the captured verdict (no key); `MODE="live"` re-runs the LLM (needs an optional NVIDIA key).
2. **Fallback -> `trapx_<DATE>.log`** if the JSONL is missing/empty for that bar -> shows the verdict the engine already rendered (display-only, cannot re-run).

The result is saved back into the same `<Mon_DD>` folder.

## 1. (Optional) NVIDIA key
Only needed for `MODE="live"` (a fresh LLM re-run). For `MODE="replay"` and the log fallback, **no key is needed** — leave it unset.

In [ ]:
!pip install -q openai
import os
NVIDIA_OK = False
key = None
try:
    from google.colab import userdata
    key = userdata.get('NVIDIA_API_KEY')
except Exception:
    pass
if key:
    os.environ['NVIDIA_API_KEY'] = key.strip()
    NVIDIA_OK = True
    print('NVIDIA key found - live re-run available (set MODE="live").')
else:
    print('No NVIDIA key - REPLAY / LOG mode only (reads captured verdicts, no LLM call). This is fine.')

## 2. Embedded skills (system prompts) + runner helpers
All 23 skill prompts are embedded here and matched to each log record by `rules_sha` (exact), falling back to `rules_chars`.

In [ ]:
SKILLS_B64 = "..."

import json, base64, hashlib, re
SKILLS = json.loads(base64.b64decode(SKILLS_B64).decode('utf-8'))   # {filename: content}
SHA2SKILL = {hashlib.sha256(v.encode('utf-8')).hexdigest()[:16]: (k, v) for k, v in SKILLS.items()}
LEN2SKILL = {}
for k, v in SKILLS.items():
    LEN2SKILL.setdefault(len(v), (k, v))
print('skills loaded:', len(SKILLS))

KNOWN_EMOJI = "✅⚠️❌🔄🟢🔴🟡🟠⚪🔵"

def parse_verdict(text):
    vline, score, action = "", None, ""
    lines = [l.rstrip() for l in text.splitlines()]
    for l in lines:
        if l.strip() and any(e in l for e in KNOWN_EMOJI):
            vline = l.strip(); break
    m = re.search(r"[Ss]core:?\s*\[?\s*([+-]?\d+(?:\.\d+)?)", text) or re.search(r"\[\s*([+-]?\d+(?:\.\d+)?)\s*\]", text)
    if m: score = float(m.group(1))
    for l in lines:
        if l.strip().lower().startswith("action"):
            action = l.strip(); break
    return vline, score, action

def select_skill(rec):
    sha = rec.get('rules_sha')
    if sha in SHA2SKILL: return SHA2SKILL[sha]
    if rec.get('rules_chars') in LEN2SKILL: return LEN2SKILL[rec['rules_chars']]
    return (None, None)

def call_nvidia(system, user, model):
    from openai import OpenAI
    cli = OpenAI(base_url='https://integrate.api.nvidia.com/v1', api_key=os.environ['NVIDIA_API_KEY'], timeout=60)
    r = cli.chat.completions.create(model=model,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            temperature=0.0, seed=42)
    return (r.choices[0].message.content or '').strip()


def parse_log_verdicts(text, bar_time):
    """Fallback: pull rendered verdicts at a bar from a trapx_<DATE>.log.
    Tracks the current 'Bar HH:MM' / 'ADVISORY @ HH:MM', collects 'Verdict:'
    lines under the target bar, dedups, and grabs the touchpoint box-title
    hint + the following Action line. Display-only (no snapshot)."""
    import re
    lines = text.splitlines()
    bar = None
    out, seen = [], set()
    bar_re = re.compile(r'Bar (\d\d:\d\d)')
    adv_re = re.compile(r'ADVISORY @ (\d\d:\d\d)')
    box_re = re.compile(r'([A-Z][A-Z -]{2,}) ADVISORY @')
    for i, raw in enumerate(lines):
        m = bar_re.search(raw) or adv_re.search(raw)
        if m:
            bar = m.group(1)
        l = raw.strip().lstrip('|').lstrip('│').strip()
        if l.startswith('Verdict:') and bar == bar_time:
            if (bar, l) in seen:
                continue
            seen.add((bar, l))
            action = ''
            for nxt in lines[i+1:i+8]:
                t = nxt.strip().lstrip('|').lstrip('│').strip()
                if t.startswith('Action'):
                    action = t
                    break
                if t.startswith(('Dtls', '└', '━')):
                    break
            hint = '(generic)'
            for cprev in lines[max(0, i-4):i]:
                tb = box_re.search(cprev)
                if tb:
                    hint = tb.group(1).strip().title()
            sm = re.search(r'\[\s*([+-]?\d+\.\d+)\s*\]', l)
            out.append({'verdict': l, 'score': float(sm.group(1)) if sm else None,
                        'action': action, 'touchpoint_hint': hint})
    return out
print('helpers ready (incl. parse_log_verdicts fallback)')

## 3. Choose the date, time, and your Drive folder

In [ ]:
#@title Parameters { run: "auto" }
DATE     = "2026-06-05"  #@param {type:"string"}
BAR_TIME = "11:06"       #@param {type:"string"}
MODE     = "replay"      #@param ["replay", "live"]
# Parent of your per-day folders in Google Drive (each named like 'Jun_05'):
VM_LOGS_DIR = "/content/drive/MyDrive/VM-4-logs"  #@param {type:"string"}

from datetime import datetime
FOLDER_NAME = datetime.strptime(DATE, "%Y-%m-%d").strftime("%b_%d")   # 2026-06-05 -> Jun_05
DAY_DIR = f"{VM_LOGS_DIR}/{FOLDER_NAME}"
print(f"DATE={DATE}  ->  folder '{FOLDER_NAME}'")
print(f"BAR_TIME={BAR_TIME}   MODE={MODE}")
print(f"DAY_DIR = {DAY_DIR}")


## 4. Mount Drive -> resolve the verdict source
Tries the **JSONL** first; if it's missing or has nothing at `BAR_TIME`, falls back to the **`trapx_<DATE>.log`** in the same folder (display-only).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')      # one-time Google auth popup

import os, glob, json
if not os.path.isdir(DAY_DIR):
    raise SystemExit("Folder not found: " + DAY_DIR +
                     "  -- check VM_LOGS_DIR and that the day folder '" + FOLDER_NAME + "' exists in Drive.")

print("Folder contents:")
for q in sorted(glob.glob(DAY_DIR + '/*')):
    print("  %-55s %12d bytes" % (os.path.basename(q), os.path.getsize(q)))

SOURCE = None
at_bar = []        # JSONL records at the bar
at_bar_log = []    # log-derived verdicts at the bar
recs = []

# --- Tier 1: JSONL (snapshot + verdict) ---
cand = os.path.join(DAY_DIR, "llm_advisory_" + DATE.replace('-', '') + ".jsonl")
if not os.path.exists(cand):
    hits = sorted(glob.glob(DAY_DIR + '/llm_advisory_*.jsonl'))
    cand = hits[0] if hits else None
if cand:
    recs = [json.loads(l) for l in open(cand, encoding='utf-8') if l.strip()]
    at_bar = [r for r in recs if r.get('bar_time') == BAR_TIME]
    if at_bar:
        SOURCE = 'jsonl'
        print("")
        print("[Tier 1] JSONL", os.path.basename(cand), "->", len(at_bar), "touchpoint(s) at", BAR_TIME)
        for r in at_bar:
            print("   -", r['touchpoint'], " captured_score=",
                  r.get('response', {}).get('parsed', {}).get('score'))

# --- Tier 2: trapx log fallback (verdict only) ---
if SOURCE is None:
    logs = sorted(glob.glob(DAY_DIR + '/trapx_*.log')) + sorted(glob.glob(DAY_DIR + '/trapx_*.log.txt'))
    if logs:
        txt = open(logs[0], encoding='utf-8', errors='replace').read()
        at_bar_log = parse_log_verdicts(txt, BAR_TIME)
        if at_bar_log:
            SOURCE = 'log'
            print("")
            print("[Tier 2] JSONL missing/empty -> engine log", os.path.basename(logs[0]),
                  "->", len(at_bar_log), "verdict(s) at", BAR_TIME, "(display-only, cannot re-run)")

# --- nothing ---
if SOURCE is None:
    avail = sorted({r.get('bar_time') for r in recs}) if recs else []
    msg = "No data for " + BAR_TIME + ". "
    msg += ("Available JSONL bars: " + str(avail)) if avail else "No llm_advisory_*.jsonl or trapx_*.log in this folder."
    raise SystemExit(msg)

print("")
print("SOURCE =", SOURCE)

## 5. Run every touchpoint at that bar → verdict board

In [ ]:
import os
from datetime import datetime
os.makedirs('/content/reports', exist_ok=True)
LOG_PATH = "/content/reports/verdict_" + DATE + "_" + BAR_TIME.replace(':', '') + ".log"
out = []
def emit(s=""):
    print(s)
    out.append(s)

emit("# Verdict - " + DATE + " bar " + BAR_TIME + " - SOURCE=" + str(SOURCE) + " - MODE=" + MODE + " - " + datetime.now().isoformat())
board = []

if SOURCE == 'jsonl':
    for r in at_bar:
        tp = r['touchpoint']
        fn, system = select_skill(r)
        user = r.get('request', {}).get('user_message', '')
        emit("")
        emit("=" * 70)
        emit("TOUCHPOINT: " + tp + "   skill=" + str(fn))
        if MODE == 'live' and NVIDIA_OK and system and user:
            try:
                text = call_nvidia(system, user, r.get('model', 'meta/llama-3.3-70b-instruct'))
                src = 'LIVE NVIDIA'
            except Exception as e:
                text = r.get('response', {}).get('raw_text', '')
                src = 'REPLAY (live failed: ' + str(e) + ')'
        else:
            text = r.get('response', {}).get('raw_text', '')
            src = 'REPLAY (captured)' if (MODE != 'live' or NVIDIA_OK) else 'REPLAY (no NVIDIA key)'
        v, s, a = parse_verdict(text)
        board.append((tp, s, v or '(no verdict line)'))
        emit("  [" + src + "]")
        for ln in (text or '(empty)').strip().splitlines():
            emit("  | " + ln)
        emit("  parsed score=" + str(s) + "   captured=" +
             str(r.get('response', {}).get('parsed', {}).get('score')))

elif SOURCE == 'log':
    emit("")
    emit("(engine-log fallback - verdicts the engine already rendered; no snapshot, cannot re-run)")
    for d in at_bar_log:
        board.append((d.get('touchpoint_hint', '(generic)'), d.get('score'), d.get('verdict')))
        emit("")
        emit("=" * 70)
        emit("TOUCHPOINT(hint): " + d.get('touchpoint_hint', '(generic)') + "   [FROM ENGINE LOG]")
        emit("  | " + d.get('verdict', ''))
        if d.get('action'):
            emit("  | " + d['action'])

emit("")
emit("=" * 70)
emit("PER-BAR VERDICT BOARD - " + DATE + " " + BAR_TIME + "  (source=" + str(SOURCE) + ")")
for tp, s, v in board:
    emit("  - " + (tp or '')[:28].ljust(28) + " score=" + str(s).ljust(6) + " " + str(v))
nums = [s for _, s, v in board if isinstance(s, (int, float))]
if nums:
    emit("  naive mean of scores = %+.3f  (NOT a negotiated verdict - see CHA-207)" % (sum(nums) / len(nums)))

with open(LOG_PATH, 'w', encoding='utf-8') as fh:
    fh.write(chr(10).join(out))
print("")
print("saved log ->", LOG_PATH)

## 6. Save the verdict log **back into the same day folder**
Writes `verdict_<DATE>_<TIME>.log` into `VM-4-logs/<Mon_DD>/` — the same folder the snapshot was read from.

In [ ]:
import shutil, os
dest = os.path.join(DAY_DIR, os.path.basename(LOG_PATH))
shutil.copy(LOG_PATH, dest)        # Drive is mounted -> writes straight to the day folder
print("saved verdict log into the source folder:")
print("  ", dest)
